# Day 2: Advanced Multi-Agent Systems for Telecom Security

## Workshop Overview

Welcome to Day 2 of the **AI-Powered Automation in Telecom Security** workshop!

### What You'll Learn Today

Today we level up from single LLM calls to **advanced multi-agent systems**:

1. **Tool Calling** - AI agents that use external functions and APIs
2. **Multi-System Alert Correlation** - Connecting dots across multiple security systems
3. **Autonomous Threat Hunting** - AI that proactively searches for threats
4. **Decision-Making Frameworks** - When should AI act autonomously?

### Yesterday's Foundation

Day 1 taught us:
- ✅ Prompt engineering (Role + Task + Constraints + Output)
- ✅ Single LLM calls solve 80% of repetitive tasks
- ✅ Immediate ROI (88-99% time savings)

### Today's Advancement

Today we tackle the **complex 20%** that requires:
- 🔄 Multiple systems coordination
- 🛠️ Tool integration
- 🤖 Agent-to-agent communication
- ⚡ Real-time decision making

---

## Setup Instructions

In [1]:
# Install dependencies
!pip install arshai -q

print("✅ Dependencies installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.1/257.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 1.3 MB/s eta 0:00:00
✅ Dependencies installed!


In [2]:
import os
import asyncio
import json
from datetime import datetime, timedelta
from typing import Dict, Any, List
import random

# Arshai imports
from arshai.core.interfaces.illm import ILLMConfig, ILLMInput
from arshai.core.interfaces.iagent import IAgentInput
from arshai.llms.openrouter import OpenRouterClient
from arshai.agents.base import BaseAgent

print("✅ Imports successful!")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_config.py:373: UserWarning: Valid config keys have changed in V2:
* 'allow_mutation' has been removed
* 'smart_union' has been removed
  warnings.warn(message, UserWarning)


✅ Imports successful!


In [3]:
# Set up API key (same as Day 1)
try:
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
    print("✅ API key loaded from Colab secrets")
except:
    if "OPENROUTER_API_KEY" in os.environ:
        print("✅ API key loaded from environment")
    else:
        print("⚠️  Please set OPENROUTER_API_KEY")

✅ API key loaded from Colab secrets


In [4]:
# Initialize LLM client
llm_config = ILLMConfig(
    model="openai/gpt-4o-mini",
    temperature=0.3,
    max_tokens=3000  # More tokens for complex analysis
)

llm_client = OpenRouterClient(llm_config)

print("✅ LLM client initialized")
print(f"   Model: {llm_config.model}")

2025-10-08 11:14:17.078 | INFO     | arshai.observability.telemetry_manager:callHandlers:1762 - Parent application mode: No explicit endpoint, will use parent OTEL setup
2025-10-08 11:14:17.079 | INFO     | arshai.observability.telemetry_manager:callHandlers:1762 - Parent application mode: No explicit endpoint, will use parent OTEL setup
2025-10-08 11:14:17.080 | INFO     | arshai.observability.telemetry_manager:callHandlers:1762 - Parent application mode: No explicit endpoint, will use parent OTEL setup
2025-10-08 11:14:17.081 | INFO     | arshai.observability.telemetry_manager:callHandlers:1762 - Using parent application's OTEL configuration
2025-10-08 11:14:17.082 | INFO     | arshai.observability.telemetry_manager:callHandlers:1762 - Using parent application's OTEL configuration
2025-10-08 11:14:17.083 | INFO     | arshai.observability.telemetry_manager:callHandlers:1762 - Using parent application's OTEL configuration
2025-10-08 11:14:17.084 | INFO     | arshai.observability.teleme

---

## Part 1: Tool Calling - Agents with External Capabilities

### Why Tool Calling?

LLMs alone can't:
- Query databases
- Fetch real-time data
- Execute system commands
- Access external APIs

**Tool calling** gives agents the ability to interact with external systems.

### How It Works

1. **Define tools** as Python functions
2. **Pass tools** to the LLM
3. **LLM decides** which tools to call
4. **Results integrated** into the response

### Example: Security System Query Agent

Let's build an agent that can query multiple security systems.

In [5]:
# Mock security system databases (in production, these would be real API calls)

# Firewall logs database
FIREWALL_LOGS = [
    {"timestamp": "2025-10-06T14:23:41Z", "source_ip": "185.220.101.47",
     "dest_ip": "10.50.20.15", "action": "blocked", "attempts": 15},
    {"timestamp": "2025-10-06T14:23:45Z", "source_ip": "185.220.101.47",
     "dest_ip": "10.50.20.16", "action": "blocked", "attempts": 12},
]

# Authentication logs database
AUTH_LOGS = [
    {"timestamp": "2025-10-06T14:15:22Z", "user": "jsmith",
     "ip": "185.220.101.47", "location": "Russia", "status": "success", "mfa": False},
    {"timestamp": "2025-10-06T14:16:10Z", "user": "jsmith",
     "ip": "185.220.101.47", "location": "Russia", "status": "mfa_failed", "mfa": True},
]

# Database query logs
DB_LOGS = [
    {"timestamp": "2025-10-06T14:17:45Z", "user": "jsmith",
     "query": "SELECT table_name FROM information_schema.tables", "rows": 127},
    {"timestamp": "2025-10-06T14:18:12Z", "user": "jsmith",
     "query": "SELECT * FROM customer_data LIMIT 10", "rows": 10},
    {"timestamp": "2025-10-06T14:19:30Z", "user": "jsmith",
     "query": "SELECT * FROM customer_data WHERE account_status='premium'", "rows": 12847},
]

# Network traffic logs
NETWORK_LOGS = [
    {"timestamp": "2025-10-06T14:20:15Z", "source_ip": "10.50.20.15",
     "dest_ip": "185.220.101.47", "protocol": "HTTPS", "data_mb": 47.3, "duration_sec": 180},
]

# Threat intelligence database
THREAT_INTEL = {
    "185.220.101.47": {
        "reputation": "malicious",
        "threat_actor": "APT-2891",
        "known_for": "Data theft, credential harvesting",
        "incidents_90d": 47,
        "first_seen": "2024-08-15"
    }
}

# User profile database
USER_PROFILES = {
    "jsmith": {
        "name": "John Smith",
        "role": "Database Administrator",
        "usual_locations": ["London, UK"],
        "vpn_usage": "rare",
        "last_30d_logins": 23,
        "all_from_uk": True
    }
}

print("✅ Mock security databases created")

✅ Mock security databases created


In [6]:
class SecurityInvestigationAgent(BaseAgent):
    """Agent that can query multiple security systems to investigate alerts."""

    def __init__(self, llm_client):
        system_prompt = """You are a senior security analyst with access to multiple security systems.

Your job is to investigate security alerts by querying relevant systems and correlating events.

Available tools:
- query_firewall_logs: Get firewall activity for an IP or time range
- query_auth_logs: Get authentication events for a user or IP
- query_database_logs: Get database queries for a user
- query_network_traffic: Get network traffic for an IP
- check_threat_intel: Look up IP reputation
- get_user_profile: Get user account information

INVESTIGATION PROCESS:
1. Start with the initial alert
2. Query relevant systems to gather context
3. Correlate events across systems
4. Identify attack patterns
5. Assess impact and provide recommendations

PROVIDE YOUR FINDINGS IN THIS FORMAT:

CORRELATED INCIDENT REPORT
Severity: [LOW/MEDIUM/HIGH/CRITICAL]
Confidence: [percentage]

ATTACK TIMELINE:
[Chronological list of correlated events]

CORRELATION ANALYSIS:
[How events are connected]

THREAT ASSESSMENT:
[What this indicates]

RECOMMENDED ACTIONS:
[Prioritized, specific steps]
"""
        super().__init__(llm_client, system_prompt)

    async def process(self, input: IAgentInput) -> Dict[str, Any]:
        """Investigate security alert using multiple tools."""

        # Define tools for querying security systems
        def query_firewall_logs(ip_address: str = None, start_time: str = None) -> List[Dict]:
            """Query firewall logs for a specific IP or time range."""
            if ip_address:
                return [log for log in FIREWALL_LOGS if log['source_ip'] == ip_address or log['dest_ip'] == ip_address]
            return FIREWALL_LOGS

        def query_auth_logs(user: str = None, ip_address: str = None) -> List[Dict]:
            """Query authentication logs for a user or IP."""
            results = AUTH_LOGS
            if user:
                results = [log for log in results if log['user'] == user]
            if ip_address:
                results = [log for log in results if log['ip'] == ip_address]
            return results

        def query_database_logs(user: str) -> List[Dict]:
            """Query database access logs for a user."""
            return [log for log in DB_LOGS if log['user'] == user]

        def query_network_traffic(ip_address: str) -> List[Dict]:
            """Query network traffic logs for an IP address."""
            return [log for log in NETWORK_LOGS if log['source_ip'] == ip_address or log['dest_ip'] == ip_address]

        def check_threat_intel(ip_address: str) -> Dict[str, Any]:
            """Check threat intelligence database for IP reputation."""
            return THREAT_INTEL.get(ip_address, {"reputation": "unknown", "info": "No threat data available"})

        def get_user_profile(username: str) -> Dict[str, Any]:
            """Get user profile and behavior information."""
            return USER_PROFILES.get(username, {"error": "User not found"})

        # Provide tools to the LLM
        tools = {
            "query_firewall_logs": query_firewall_logs,
            "query_auth_logs": query_auth_logs,
            "query_database_logs": query_database_logs,
            "query_network_traffic": query_network_traffic,
            "check_threat_intel": check_threat_intel,
            "get_user_profile": get_user_profile
        }

        llm_input = ILLMInput(
            system_prompt=self.system_prompt,
            user_message=input.message,
            regular_functions=tools
        )

        result = await self.llm_client.chat(llm_input)

        return {
            "investigation_report": result.get('llm_response', ''),
            "tools_used": list(tools.keys()),
            "timestamp": datetime.now().isoformat()
        }

print("✅ SecurityInvestigationAgent created")

✅ SecurityInvestigationAgent created


### Test the Investigation Agent

In [7]:
# Create investigation agent
investigation_agent = SecurityInvestigationAgent(llm_client)

# Initial alert
ALERT = """FIREWALL ALERT
Alert ID: FW-47821
Timestamp: 2025-10-06T14:23:41Z
Event: Multiple failed connection attempts
Source IP: 185.220.101.47
Target: 10.50.20.15 (Internal web server)
Attempts: 15
Severity: MEDIUM

INVESTIGATION REQUEST:
Investigate this alert. Use available tools to:
1. Check if this IP has other activity in our systems
2. Look up the IP reputation
3. Check for any successful authentication attempts
4. If you find compromised accounts, check their database activity
5. Check for data exfiltration
6. Correlate all events into a complete attack timeline
"""

print("\n" + "="*80)
print("MULTI-SYSTEM SECURITY INVESTIGATION")
print("="*80)
print("\nINITIAL ALERT:")
print("-" * 80)
print(ALERT)
print("\n" + "-" * 80)
print("STARTING INVESTIGATION...")
print("The agent will now query multiple systems to gather context...")
print("-" * 80)

# Run investigation
agent_input = IAgentInput(message=ALERT)
result = await investigation_agent.process(agent_input)

print("\n" + "="*80)
print("INVESTIGATION REPORT")
print("="*80)
print(result['investigation_report'])
print("\n" + "-" * 80)
print(f"Tools available: {', '.join(result['tools_used'])}")
print(f"Investigation completed at: {result['timestamp']}")
print("="*80)


MULTI-SYSTEM SECURITY INVESTIGATION

INITIAL ALERT:
--------------------------------------------------------------------------------
FIREWALL ALERT
Alert ID: FW-47821
Timestamp: 2025-10-06T14:23:41Z
Event: Multiple failed connection attempts
Source IP: 185.220.101.47
Target: 10.50.20.15 (Internal web server)
Attempts: 15
Severity: MEDIUM

INVESTIGATION REQUEST:
Investigate this alert. Use available tools to:
1. Check if this IP has other activity in our systems
2. Look up the IP reputation
3. Check for any successful authentication attempts
4. If you find compromised accounts, check their database activity
5. Check for data exfiltration
6. Correlate all events into a complete attack timeline


--------------------------------------------------------------------------------
STARTING INVESTIGATION...
The agent will now query multiple systems to gather context...
--------------------------------------------------------------------------------
2025-10-08 11:14:18.370 | INFO     | arshai.O

### 🎯 Key Observations

Notice how the agent:
1. **Decides which tools to call** based on the investigation needs
2. **Calls tools in logical order** (check threat intel → check auth logs → check database logs → check network traffic)
3. **Correlates findings** across multiple systems
4. **Builds a timeline** of the attack
5. **Provides actionable recommendations**

**Manual Process:**
- Query each system separately
- Manually correlate events
- Time: 2-3 hours

**AI Agent Process:**
- Queries all systems automatically
- Correlates intelligently
- Time: 30-60 seconds

---

## Part 2: Advanced Example - Multi-Agent Threat Hunting System

### The Challenge

Traditional threat hunting requires:
- Expert analysts generating hypotheses
- Manual data collection from multiple sources
- Hours of analysis per hunt
- Limited by analyst availability (8 hours/day)

### The Solution: Autonomous Threat Hunting

A multi-agent system that:
1. **Hypothesis Generator Agent** - Creates threat hunting hypotheses
2. **Data Collection Agent** - Gathers evidence
3. **Analysis Agent** - Investigates patterns
4. **Reporting Agent** - Documents findings

Let's build this system!

In [ ]:
# First, let's create mock data for threat hunting

# User activity data
USER_ACTIVITY = {
    "ajenkins": {
        "name": "Alice Jenkins",
        "role": "Customer Service Representative",
        "department": "Customer Support",
        "tenure_years": 2.3,
        "normal_cloud_usage_mb_month": 15,
        "normal_work_hours": "09:00-17:00",
        "cloud_uploads_last_30d": [
            {"date": "2025-09-15", "time": "23:47", "service": "OneDrive", "size_mb": 47, "file_type": ".zip"},
            {"date": "2025-09-17", "time": "02:15", "service": "OneDrive", "size_mb": 52, "file_type": ".zip"},
            {"date": "2025-09-20", "time": "01:33", "service": "OneDrive", "size_mb": 45, "file_type": ".zip"},
            {"date": "2025-09-23", "time": "03:22", "service": "OneDrive", "size_mb": 38, "file_type": ".zip"},
            {"date": "2025-09-26", "time": "02:50", "service": "OneDrive", "size_mb": 41, "file_type": ".zip"},
            {"date": "2025-09-29", "time": "01:15", "service": "OneDrive", "size_mb": 49, "file_type": ".zip"},
            {"date": "2025-10-02", "time": "02:40", "service": "OneDrive", "size_mb": 44, "file_type": ".zip"},
            {"date": "2025-10-05", "time": "01:55", "service": "OneDrive", "size_mb": 46, "file_type": ".zip"},
        ],
        "auth_anomalies": [
            {"date": "2025-09-14", "event": "password_changed", "device": "personal_laptop"},
        ],
        "database_queries_daily": 350,
        "normal_database_queries_daily": 50
    },
    "bmiller": {
        "name": "Bob Miller",
        "role": "Senior Developer",
        "department": "Engineering",
        "tenure_years": 5.8,
        "normal_cloud_usage_mb_month": 450,
        "normal_work_hours": "10:00-19:00",
        "cloud_uploads_last_30d": [
            {"date": "2025-09-18", "time": "14:30", "service": "GitHub", "size_mb": 120, "file_type": ".git"},
            {"date": "2025-09-25", "time": "16:45", "service": "GitHub", "size_mb": 85, "file_type": ".git"},
        ],
        "auth_anomalies": [],
        "database_queries_daily": 80,
        "normal_database_queries_daily": 75
    }
}

print("✅ User activity database created")

In [ ]:
class HypothesisGeneratorAgent(BaseAgent):
    """Generates threat hunting hypotheses based on threat intelligence."""

    def __init__(self, llm_client):
        system_prompt = """You are a threat intelligence analyst who creates threat hunting hypotheses.

Your job is to generate specific, actionable threat hunting hypotheses based on:
- Recent threat intelligence
- Industry trends
- Known attack patterns
- Company vulnerabilities

For each hypothesis, provide:
1. HYPOTHESIS: Clear statement of what to hunt for
2. RATIONALE: Why this threat is relevant
3. HUNT STRATEGY: How to look for it
4. SUCCESS CRITERIA: What finding would confirm the threat
5. PRIORITY: [LOW/MEDIUM/HIGH/CRITICAL]

Focus on realistic, practical threats for a telecom company.
"""
        super().__init__(llm_client, system_prompt)

    async def process(self, input: IAgentInput) -> Dict[str, Any]:
        llm_input = ILLMInput(
            system_prompt=self.system_prompt,
            user_message=input.message
        )

        result = await self.llm_client.chat(llm_input)

        return {
            "hypothesis": result.get('llm_response', ''),
            "generated_at": datetime.now().isoformat()
        }


class ThreatHunterAgent(BaseAgent):
    """Executes threat hunts by analyzing user activity data."""

    def __init__(self, llm_client):
        system_prompt = """You are an expert threat hunter analyzing user behavior for security threats.

INVESTIGATION APPROACH:
1. Review hypothesis
2. Use available tools to gather user activity data
3. Identify anomalies and patterns
4. Correlate multiple indicators
5. Assess threat likelihood

REPORT FORMAT:

THREAT HUNT REPORT
Hypothesis: [restate hypothesis]
Status: [THREAT CONFIRMED / SUSPICIOUS / NO THREAT FOUND]
Confidence: [percentage]

FINDINGS:
[Detailed findings with specific evidence]

SUSPICIOUS ACCOUNTS:
[List accounts with risk scores]

RED FLAGS IDENTIFIED:
[Specific suspicious behaviors]

RECOMMENDED ACTIONS:
[Immediate and strategic actions]

URGENCY: [LOW/MEDIUM/HIGH/CRITICAL]
"""
        super().__init__(llm_client, system_prompt)

    async def process(self, input: IAgentInput) -> Dict[str, Any]:
        """Execute threat hunt with tool access."""

        # Define hunting tools
        def get_user_cloud_activity(username: str) -> Dict[str, Any]:
            """Get user's cloud storage upload activity."""
            if username in USER_ACTIVITY:
                user = USER_ACTIVITY[username]
                return {
                    "user": username,
                    "role": user['role'],
                    "normal_usage_mb_month": user['normal_cloud_usage_mb_month'],
                    "uploads_last_30d": user['cloud_uploads_last_30d'],
                    "total_uploaded_mb": sum(u['size_mb'] for u in user['cloud_uploads_last_30d'])
                }
            return {"error": "User not found"}

        def get_user_auth_history(username: str) -> Dict[str, Any]:
            """Get user's authentication history and anomalies."""
            if username in USER_ACTIVITY:
                user = USER_ACTIVITY[username]
                return {
                    "user": username,
                    "normal_work_hours": user['normal_work_hours'],
                    "auth_anomalies": user['auth_anomalies']
                }
            return {"error": "User not found"}

        def get_user_database_activity(username: str) -> Dict[str, Any]:
            """Get user's database query patterns."""
            if username in USER_ACTIVITY:
                user = USER_ACTIVITY[username]
                return {
                    "user": username,
                    "role": user['role'],
                    "current_queries_daily": user['database_queries_daily'],
                    "normal_queries_daily": user['normal_database_queries_daily'],
                    "deviation_percent": ((user['database_queries_daily'] - user['normal_database_queries_daily']) / user['normal_database_queries_daily'] * 100)
                }
            return {"error": "User not found"}

        def list_all_users() -> List[str]:
            """Get list of all users to investigate."""
            return list(USER_ACTIVITY.keys())

        def get_user_profile(username: str) -> Dict[str, Any]:
            """Get complete user profile."""
            if username in USER_ACTIVITY:
                return USER_ACTIVITY[username]
            return {"error": "User not found"}

        # Provide tools
        tools = {
            "get_user_cloud_activity": get_user_cloud_activity,
            "get_user_auth_history": get_user_auth_history,
            "get_user_database_activity": get_user_database_activity,
            "list_all_users": list_all_users,
            "get_user_profile": get_user_profile
        }

        llm_input = ILLMInput(
            system_prompt=self.system_prompt,
            user_message=input.message,
            regular_functions=tools
        )

        result = await self.llm_client.chat(llm_input)

        return {
            "hunt_report": result.get('llm_response', ''),
            "tools_available": list(tools.keys()),
            "hunt_completed_at": datetime.now().isoformat()
        }

print("✅ Threat hunting agents created")

### Execute Autonomous Threat Hunt

In [ ]:
# Step 1: Generate hypothesis
print("\n" + "="*80)
print("AUTONOMOUS THREAT HUNTING SYSTEM")
print("="*80)

print("\n" + "-" * 80)
print("STEP 1: HYPOTHESIS GENERATION")
print("-" * 80)

hypothesis_agent = HypothesisGeneratorAgent(llm_client)

hypothesis_request = """Generate a threat hunting hypothesis for data exfiltration via cloud storage.

Context:
- We're a telecommunications company with customer data
- Employees use OneDrive, Dropbox, and Google Drive
- Recent threat intel shows insiders using cloud storage for slow exfiltration
- Focus on detecting unusual patterns that might indicate data theft
"""

hypothesis_input = IAgentInput(message=hypothesis_request)
hypothesis_result = await hypothesis_agent.process(hypothesis_input)

print(hypothesis_result['hypothesis'])
print(f"\n⏱️  Generated at: {hypothesis_result['generated_at']}")

In [ ]:
# Step 2: Execute threat hunt
print("\n" + "-" * 80)
print("STEP 2: EXECUTING THREAT HUNT")
print("-" * 80)
print("\nThe hunting agent will now:")
print("1. List all users")
print("2. Check cloud storage activity for each user")
print("3. Identify anomalous patterns")
print("4. Correlate with authentication and database activity")
print("5. Generate findings report")
print("-" * 80)

hunter_agent = ThreatHunterAgent(llm_client)

hunt_request = f"""Execute the following threat hunt:

HYPOTHESIS:
Insiders may be using cloud storage services (OneDrive, Dropbox, Google Drive) to slowly exfiltrate data.

INSTRUCTIONS:
1. Use list_all_users() to get all users
2. For each user, check their cloud activity using get_user_cloud_activity()
3. Look for:
   - Uploads during non-business hours (nights, weekends)
   - Sudden increase in upload volume
   - Unusual file types (especially .zip files)
   - Regular patterns (automated uploads)
4. For suspicious users, check auth history and database activity
5. Provide a complete threat assessment

Start your investigation now using the available tools.
"""

hunt_input = IAgentInput(message=hunt_request)
hunt_result = await hunter_agent.process(hunt_input)

print("\n" + "="*80)
print("THREAT HUNT RESULTS")
print("="*80)
print(hunt_result['hunt_report'])
print("\n" + "-" * 80)
print(f"Tools available: {', '.join(hunt_result['tools_available'])}")
print(f"Hunt completed at: {hunt_result['hunt_completed_at']}")
print("="*80)

### 💰 ROI Calculation: Autonomous Threat Hunting

In [ ]:
print("\n" + "="*80)
print("ROI CALCULATION: AUTONOMOUS THREAT HUNTING")
print("="*80)

# Manual threat hunting
manual_hunt_time_hours = 8  # Average time for one hunt
analyst_cost_per_hour = 75  # Senior analyst rate
manual_hunts_per_week = 2  # Limited by analyst time
manual_cost_per_hunt = manual_hunt_time_hours * analyst_cost_per_hour

# AI threat hunting
ai_hunt_time_minutes = 3  # Automated execution
ai_review_time_minutes = 15  # Human review of findings
ai_total_time_hours = (ai_hunt_time_minutes + ai_review_time_minutes) / 60
ai_hunts_per_day = 10  # Can run continuously
ai_hunts_per_week = ai_hunts_per_day * 7
ai_cost_per_hunt = (ai_review_time_minutes / 60) * analyst_cost_per_hour

# Savings
time_savings_per_hunt = manual_hunt_time_hours - ai_total_time_hours
cost_savings_per_hunt = manual_cost_per_hunt - ai_cost_per_hunt
additional_hunts_per_week = ai_hunts_per_week - manual_hunts_per_week
weekly_savings = manual_hunts_per_week * cost_savings_per_hunt
yearly_savings = weekly_savings * 52

# Detection improvement
insider_threat_avg_detection_days = 287  # Industry average
ai_avg_detection_days = 21  # With daily automated hunts
detection_improvement = ((insider_threat_avg_detection_days - ai_avg_detection_days) / insider_threat_avg_detection_days) * 100

print(f"\n📊 MANUAL THREAT HUNTING:")
print(f"   Time per hunt: {manual_hunt_time_hours} hours")
print(f"   Cost per hunt: £{manual_cost_per_hunt:,.2f}")
print(f"   Hunts per week: {manual_hunts_per_week}")
print(f"   Limited by: Analyst availability")

print(f"\n🤖 AI-POWERED THREAT HUNTING:")
print(f"   Execution time: {ai_hunt_time_minutes} minutes (automated)")
print(f"   Review time: {ai_review_time_minutes} minutes (human)")
print(f"   Total time: {ai_total_time_hours:.1f} hours")
print(f"   Cost per hunt: £{ai_cost_per_hunt:,.2f}")
print(f"   Hunts per week: {ai_hunts_per_week} (24/7 operation)")

print(f"\n💰 SAVINGS:")
print(f"   Time saved per hunt: {time_savings_per_hunt:.1f} hours ({(time_savings_per_hunt/manual_hunt_time_hours*100):.1f}%)")
print(f"   Cost saved per hunt: £{cost_savings_per_hunt:,.2f}")
print(f"   Additional hunts per week: {additional_hunts_per_week}")
print(f"   Weekly cost savings: £{weekly_savings:,.2f}")
print(f"   Yearly cost savings: £{yearly_savings:,.2f}")

print(f"\n📈 SECURITY IMPROVEMENT:")
print(f"   Manual detection time: {insider_threat_avg_detection_days} days (industry average)")
print(f"   AI detection time: {ai_avg_detection_days} days")
print(f"   Detection improvement: {detection_improvement:.1f}% faster")
print(f"   Coverage: {(ai_hunts_per_week/manual_hunts_per_week):.0f}x more hunts executed")
print(f"   Availability: 24/7 (vs. business hours only)")

print("\n" + "="*80)

---

## Part 3: Decision-Making Frameworks for Autonomous Action

### The Challenge: When Should AI Act Autonomously?

Not all threats should be handled automatically. We need frameworks for deciding when AI should:
- **Act immediately** (autonomous response)
- **Recommend action** (human approval required)
- **Escalate** (expert review needed)

### Decision Matrix

Let's build an agent that makes intelligent decisions about response actions.

In [ ]:
class ResponseDecisionAgent(BaseAgent):
    """Agent that decides appropriate response actions based on threat assessment."""

    def __init__(self, llm_client):
        system_prompt = """You are a senior security operations decision-maker.

Your job is to analyze threats and recommend appropriate response levels.

DECISION FRAMEWORK:

AUTONOMOUS ACTION (immediate, no approval needed):
- Confidence > 95%
- Business impact: LOW to MEDIUM
- Well-known attack patterns
- Reversible actions (can undo if wrong)
- Time-critical (DDoS, ransomware)

RECOMMENDED ACTION (prepare response, await approval):
- Confidence > 85%
- Business impact: MEDIUM to HIGH
- Established procedures exist
- May affect operations

ESCALATE TO HUMAN (expert review required):
- Confidence < 85%
- Business impact: HIGH to CRITICAL
- Novel attack patterns
- Irreversible actions
- Legal/compliance implications

PROVIDE YOUR DECISION IN THIS FORMAT:

RESPONSE DECISION

DECISION: [AUTONOMOUS_ACTION / RECOMMENDED_ACTION / ESCALATE_TO_HUMAN]

CONFIDENCE LEVEL: [percentage]

BUSINESS IMPACT: [LOW / MEDIUM / HIGH / CRITICAL]

REASONING:
[Explain decision using the framework]

RECOMMENDED ACTIONS:
[Specific steps, prioritized]

APPROVAL REQUIRED FROM: [None / SOC Manager / CISO / Executive]

ESTIMATED RESPONSE TIME: [time estimate]

REVERSIBILITY: [Fully reversible / Partially reversible / Irreversible]
"""
        super().__init__(llm_client, system_prompt)

    async def process(self, input: IAgentInput) -> Dict[str, Any]:
        llm_input = ILLMInput(
            system_prompt=self.system_prompt,
            user_message=input.message
        )

        result = await self.llm_client.chat(llm_input)

        return {
            "decision": result.get('llm_response', ''),
            "timestamp": datetime.now().isoformat()
        }

print("✅ ResponseDecisionAgent created")

### Test Decision Framework with Different Scenarios

In [ ]:
decision_agent = ResponseDecisionAgent(llm_client)

# Test scenarios
SCENARIOS = [
    {
        "name": "DDoS Attack",
        "threat": """THREAT ASSESSMENT:
Type: DDoS Attack (SYN Flood)
Confidence: 97%
Attack Volume: 45 Gbps (18x normal traffic)
Attack Sources: 847 unique IPs (234 known botnet IPs)
Target: Customer-facing web servers
Impact: 2,847 customers affected, 73% service degradation
Duration: Ongoing (23 minutes)
Pattern Match: Known DDoS signature (100% match)

PROPOSED RESPONSE:
1. Activate rate limiting
2. Enable DDoS scrubbing service
3. Block top 50 attack source IPs
4. Auto-scale infrastructure

All actions are reversible."""
    },
    {
        "name": "Data Exfiltration",
        "threat": """THREAT ASSESSMENT:
Type: Suspected Data Exfiltration
Confidence: 89%
User: ajenkins (Customer Service Rep)
Activity: 362 MB uploaded to OneDrive over 21 days
Pattern: Night-time uploads (1-3 AM), .zip files
Anomaly: 181x normal monthly usage
Red Flags: Password changed before exfiltration began, no SMS verification response
Impact: ~36,000 customer records potentially compromised

PROPOSED RESPONSE:
1. Disable user account
2. Revoke all active sessions
3. Block access to customer database
4. Contact Microsoft to freeze OneDrive account
5. Begin forensic investigation

Account suspension affects one employee."""
    },
    {
        "name": "Novel Attack Pattern",
        "threat": """THREAT ASSESSMENT:
Type: Unknown - Novel Attack Pattern
Confidence: 68%
Observations:
- Unusual API calls from multiple internal systems
- Pattern doesn't match known attack signatures
- Could be new vulnerability exploitation OR legitimate new application
- Affects core billing infrastructure
- No clear attribution

PROPOSED RESPONSE:
1. Isolate affected systems (would impact billing operations)
2. Block suspicious traffic patterns
3. Conduct forensic analysis

Actions would shut down billing for 2-4 hours, affecting revenue collection."""
    }
]

print("\n" + "="*80)
print("AUTONOMOUS DECISION-MAKING FRAMEWORK")
print("="*80)

for i, scenario in enumerate(SCENARIOS, 1):
    print(f"\n{'-'*80}")
    print(f"SCENARIO {i}: {scenario['name']}")
    print("-" * 80)
    print("\nTHREAT DETAILS:")
    print(scenario['threat'])
    print("\n" + "-" * 80)
    print("DECISION ANALYSIS:")
    print("-" * 80)

    agent_input = IAgentInput(message=scenario['threat'])
    result = await decision_agent.process(agent_input)

    print(result['decision'])
    print(f"\n⏱️  Decision made at: {result['timestamp']}")

print("\n" + "="*80)

### 🎯 Key Observations

Notice how the decision agent:

1. **DDoS Attack**: Likely recommended AUTONOMOUS_ACTION because:
   - Very high confidence (97%)
   - Well-known attack pattern
   - Time-critical
   - All actions are reversible
   - Medium business impact

2. **Data Exfiltration**: Likely recommended RECOMMENDED_ACTION because:
   - High confidence (89%)
   - Account suspension is significant
   - May need legal involvement
   - GDPR implications

3. **Novel Attack**: Likely recommended ESCALATE_TO_HUMAN because:
   - Lower confidence (68%)
   - Unknown pattern
   - High business impact (billing shutdown)
   - Irreversible operational impact

---

## Part 4: Building Your Own Multi-Agent System

### Exercise: Complete Incident Response System

**Your Task**: Design a multi-agent system that handles a complete incident response workflow.

**Components**:
1. **Triage Agent** - Assesses alert severity
2. **Investigation Agent** - Gathers evidence (you built this!)
3. **Decision Agent** - Determines response level (you built this!)
4. **Action Agent** - Executes or recommends actions
5. **Documentation Agent** - Creates incident reports

Let's build the missing pieces:

In [ ]:
# TODO: Build the complete system

class TriageAgent(BaseAgent):
    """Initial triage of security alerts."""

    def __init__(self, llm_client):
        system_prompt = """You are a security alert triage specialist.

Quickly assess alerts and determine:
1. SEVERITY: [LOW / MEDIUM / HIGH / CRITICAL]
2. CATEGORY: [Authentication / Network / Data / Malware / Fraud / Other]
3. INITIAL CONFIDENCE: [percentage]
4. REQUIRES_INVESTIGATION: [YES / NO]
5. URGENCY: [Can wait / Within 1 hour / Immediate]
6. ONE-LINE SUMMARY: [What is this?]

Be fast and decisive. Your triage determines what happens next.
"""
        super().__init__(llm_client, system_prompt)

    async def process(self, input: IAgentInput) -> Dict[str, Any]:
        llm_input = ILLMInput(
            system_prompt=self.system_prompt,
            user_message=input.message
        )
        result = await self.llm_client.chat(llm_input)
        return {
            "triage": result.get('llm_response', ''),
            "timestamp": datetime.now().isoformat()
        }


class DocumentationAgent(BaseAgent):
    """Creates comprehensive incident documentation."""

    def __init__(self, llm_client):
        system_prompt = """You are an incident documentation specialist.

Create complete incident reports that include:

INCIDENT REPORT
Incident ID: [auto-generated]
Date: [timestamp]
Severity: [from investigation]

EXECUTIVE SUMMARY:
[2-3 sentences for executives]

TIMELINE:
[Chronological events]

INVESTIGATION FINDINGS:
[Key discoveries]

ACTIONS TAKEN:
[What was done]

BUSINESS IMPACT:
[Customers affected, data compromised, downtime, etc.]

LESSONS LEARNED:
[What to improve]

FOLLOW-UP ACTIONS:
[Next steps]

Format for compliance and audit requirements.
"""
        super().__init__(llm_client, system_prompt)

    async def process(self, input: IAgentInput) -> Dict[str, Any]:
        llm_input = ILLMInput(
            system_prompt=self.system_prompt,
            user_message=input.message
        )
        result = await self.llm_client.chat(llm_input)
        return {
            "documentation": result.get('llm_response', ''),
            "timestamp": datetime.now().isoformat()
        }

print("✅ Additional agents created")
print("\nYou now have all components for a complete incident response system!")

### Test Complete Incident Response Workflow

In [ ]:
# Orchestrate complete workflow
print("\n" + "="*80)
print("COMPLETE INCIDENT RESPONSE WORKFLOW")
print("="*80)

# Sample alert
INCOMING_ALERT = """ALERT: Multiple Failed Login Attempts
Source: Authentication System
Time: 2025-10-06 14:23:41
User: admin
IP: 185.220.101.47
Failed attempts: 15 in 5 minutes
Previous successful login: UK (2 hours ago)
Current location: Russia
"""

# Step 1: Triage
print("\n" + "-" * 80)
print("STEP 1: TRIAGE")
print("-" * 80)
triage_agent = TriageAgent(llm_client)
triage_result = await triage_agent.process(IAgentInput(message=INCOMING_ALERT))
print(triage_result['triage'])

# Step 2: Investigation (reuse the agent we built earlier)
print("\n" + "-" * 80)
print("STEP 2: INVESTIGATION")
print("-" * 80)
print("Using SecurityInvestigationAgent to gather evidence...\n")
investigation_result = await investigation_agent.process(IAgentInput(message=INCOMING_ALERT))
print(investigation_result['investigation_report'][:500] + "...\n[truncated for display]")

# Step 3: Decision
print("\n" + "-" * 80)
print("STEP 3: RESPONSE DECISION")
print("-" * 80)
decision_input = f"""Based on this investigation:

{investigation_result['investigation_report']}

Determine the appropriate response level and actions.
"""
decision_result = await decision_agent.process(IAgentInput(message=decision_input))
print(decision_result['decision'])

# Step 4: Documentation
print("\n" + "-" * 80)
print("STEP 4: DOCUMENTATION")
print("-" * 80)
doc_agent = DocumentationAgent(llm_client)
doc_input = f"""Create comprehensive incident documentation based on:

INITIAL ALERT:
{INCOMING_ALERT}

TRIAGE:
{triage_result['triage']}

INVESTIGATION:
{investigation_result['investigation_report'][:1000]}

DECISION:
{decision_result['decision']}
"""
doc_result = await doc_agent.process(IAgentInput(message=doc_input))
print(doc_result['documentation'])

print("\n" + "="*80)
print("✅ COMPLETE INCIDENT RESPONSE WORKFLOW EXECUTED")
print("="*80)
print("\nWorkflow Stages:")
print("1. ✅ Triage (3 seconds)")
print("2. ✅ Investigation (45 seconds)")
print("3. ✅ Decision (5 seconds)")
print("4. ✅ Documentation (10 seconds)")
print("\n📊 Total time: ~63 seconds (vs. 2-4 hours manual)")
print("⚡ Time savings: 98%+")

---

## Summary: Day 2 Learnings

### What We Built Today

1. **Tool-Calling Agents**
   - Agents that query external systems
   - Multi-system correlation
   - Evidence gathering automation

2. **Multi-Agent Threat Hunting**
   - Hypothesis generation
   - Automated data collection
   - Pattern detection
   - 24/7 continuous hunting

3. **Decision-Making Frameworks**
   - When to act autonomously
   - When to escalate to humans
   - Risk vs. confidence matrices

4. **Complete Incident Response System**
   - Triage → Investigation → Decision → Documentation
   - End-to-end automation
   - 98%+ time savings

### Key Architectural Patterns

✅ **Agent Specialization** - Each agent has a specific role

✅ **Tool Integration** - Agents use external functions/APIs

✅ **Sequential Workflows** - Agents work in pipelines

✅ **Decision Hierarchies** - Confidence-based escalation

✅ **Human-in-the-Loop** - Critical decisions require approval

### ROI Summary

| Use Case | Manual Time | AI Time | Savings |
|----------|-------------|---------|----------|
| Alert Correlation | 2-3 hours | 45 sec | 99% |
| Threat Hunting | 8 hours | 3 min | 99% |
| Incident Response | 2-4 hours | 63 sec | 98% |

**Additional Benefits:**
- 24/7 operation (no human fatigue)
- Consistent quality
- Instant availability
- Scalable to any volume
- Complete audit trail

---

## What's Next?

### Implementing in Your Organization

1. **Start Small**
   - Pick one use case from Day 1 (email analysis, log contextualization)
   - Implement in pilot mode
   - Measure ROI

2. **Build Trust**
   - Run AI in parallel with manual process
   - Compare results
   - Gather analyst feedback

3. **Expand Gradually**
   - Add tool integration
   - Implement multi-agent workflows
   - Increase automation levels

4. **Monitor and Improve**
   - Track false positives
   - Refine prompts
   - Update decision thresholds

### Technical Considerations

- **API Keys**: Secure management (use environment variables, secrets managers)
- **Rate Limits**: Monitor LLM API usage
- **Cost Control**: Set budgets and alerts
- **Error Handling**: Graceful degradation
- **Audit Logging**: Track all AI decisions
- **Compliance**: GDPR, data residency, etc.

### Advanced Topics (Beyond Workshop)

- **Memory Systems**: Long-term context and learning
- **Fine-tuning**: Custom models for specific threats
- **Feedback Loops**: Continuous improvement
- **Multi-modal AI**: Image analysis (CCTV, screenshots)
- **Integration**: SIEM, SOAR, ticketing systems

---

## Final Exercise: Design Your System

**Challenge**: Design a multi-agent system for a security problem in YOUR organization.

**Questions to Consider:**
1. What repetitive task takes the most time?
2. What data sources would agents need to access?
3. How many agents would you need? What are their roles?
4. What decisions should be autonomous vs. requiring approval?
5. How would you measure success?

**Template:**

```
PROBLEM: [What security challenge are you solving?]

AGENTS:
1. [Agent Name] - [Role]
2. [Agent Name] - [Role]
...

DATA SOURCES:
- [System 1]
- [System 2]
...

WORKFLOW:
1. [Step]
2. [Step]
...

DECISION POINTS:
- Autonomous: [When?]
- Human approval: [When?]
- Escalation: [When?]

EXPECTED ROI:
- Time savings: [estimate]
- Cost savings: [estimate]
- Quality improvement: [how?]
```

---

## Resources

- **Arshai Framework**: https://github.com/felesh-ai/arshai
- **OpenRouter**: https://openrouter.ai/
- **Workshop Materials**: [Your workshop repository]
- **Security Frameworks**: NIST, MITRE ATT&CK
- **Telecom Fraud**: CFCA resources

---

## Thank You!

You've completed the **AI-Powered Automation in Telecom Security** workshop!

### You Now Know How To:

✅ Engineer effective prompts for security use cases

✅ Build single-call LLM agents for repetitive tasks

✅ Create multi-agent systems with tool integration

✅ Implement autonomous threat hunting

✅ Design decision frameworks for AI autonomy

✅ Calculate ROI and business impact

✅ Use Arshai framework for production systems

### Next Steps:

1. Experiment with these notebooks
2. Adapt examples to your use cases
3. Build a pilot system
4. Share results with your team
5. Keep learning and improving!

**Questions? Feedback? Connect with the community!**

🚀 **Good luck building the future of telecom security!**